# 01: Site EDA and physics baseline

**Question:** Can a calibrated PVWatts model, driven by the site's own irradiance sensor, predict its output well enough to spot problems?

**Before running:**
1. `python -m solarmon.ingest download 2107`
2. `python -m solarmon.inspect 2107`, then fill in `columns:` for 2107 in `config/sites.yaml`

Change `SYSTEM_ID` below to repeat this for any site.

In [ ]:
import sys
from pathlib import Path

# Make `solarmon` importable even if the package isn't pip-installed in this kernel
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else str(Path.cwd() / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from solarmon.config import get_site
from solarmon.load import load_site
from solarmon.expected import expected_ac_kw, rmse_pct
from solarmon.metrics import daily_performance
from solarmon.qa import clean, lag_scan, hourly_profile

SYSTEM_ID = 2107
site = get_site(SYSTEM_ID)
site

## 1. Load and sanity-check

Things to look for:
- **Timestamps:** what's the interval (1, 5, or 15 min)? Are there big gaps?
- **Units:** does peak power land near the nameplate (`dc_capacity_kw`)? Peak POA should be around 1,000–1,200 W/m².
- **Night values:** power should be ~0 at night. Negative values usually mean inverter standby draw.

In [ ]:
df = clean(load_site(site), site)  # impossible values (meter spikes etc.) -> NaN
print(df.index.min(), "->", df.index.max())
print("median interval:", df.index.to_series().diff().median())
df.describe().T

In [ ]:
# Data coverage by month: holes here are data gaps, not underperformance
df.notna().resample("MS").mean().plot(figsize=(12, 3), title="Fraction of intervals with data, by month")
plt.ylim(0, 1.05);

### Timestamp alignment check

Power and irradiance have to be on the same clock, or the model will look wrong even when the site is fine. Typical culprits: one file logged in UTC, a daylight-saving mismatch between files, or interval-start vs interval-end stamps.

`lag_scan` slides irradiance back and forth by up to ±12 h and finds the shift where it best matches power. **A healthy result is `best_lag_h` = 0 for every row.**

- A constant nonzero lag across all years means one file is on a different clock.
- Different lags in different years means some files changed time basis.
- A lag in only one season means a daylight-saving mismatch.

In [ ]:
print("By year:")
display(lag_scan(df, by="year"))
print("By season:")
display(lag_scan(df, by="dst"))

In [ ]:
# Visual check: if clocks agree, the two curves peak at the same hour and overlap
ax = hourly_profile(df).plot(figsize=(9, 3.5), marker="o", title="Average shape by hour of day (scaled to peak)")
ax.set(xlabel="hour (timestamp clock)", ylabel="fraction of peak");

## 2. Power vs irradiance

For a healthy array this should look like a tight, nearly linear band. Expect:
- a fan-out at high irradiance from temperature (hot panels make less power)
- a flat ceiling where the inverter clips
- points stuck near zero at high irradiance, which are outages or curtailment

These outlier patterns are exactly what the detector will hunt for in phase 2.

In [ ]:
s = df.dropna(subset=["poa_irradiance", "ac_power_kw"]).sample(min(50_000, len(df)), random_state=0)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(s["poa_irradiance"], s["ac_power_kw"], s=2, alpha=0.2)
ax.axhline(site.dc_capacity_kw, ls="--", c="gray", label="DC nameplate")
ax.set(xlabel="POA irradiance (W/m²)", ylabel="AC power (kW)", title=f"{site.name}: power vs sunlight")
ax.legend();

## 3. Expected output

Pick a **reference window** where the site looks healthy (no gaps or obvious outages in the plots above). The model calibrates to it. The default is the first 60 days.

The scale factor should land around **0.75–1.0**. Far outside that usually means a wrong column or unit.

In [ ]:
REFERENCE = None  # e.g. slice("2018-03-01", "2018-05-31")

out = expected_ac_kw(df, site, reference=REFERENCE)
print(f"calibration scale: {out.attrs['scale']:.3f}  (reference {out.attrs['reference']})")
print(f"nRMSE: {rmse_pct(out['ac_power_kw'], out['expected_ac_kw'], out['poa_irradiance']):.1f}%")

In [ ]:
# Look at a week up close: actual should track expected through clouds and all
week = out.loc[out.index[len(out) // 2]:].iloc[: 7 * 24 * 12]
ax = week[["ac_power_kw", "expected_ac_kw"]].plot(figsize=(14, 4), title="Actual vs expected, one week")
ax.set_ylabel("kW");

## 4. Daily performance index

PI = actual ÷ expected energy each day. A healthy site hovers near 1.0 whatever the weather. Sudden drops, slow slides, and step changes are the events phase 2 will detect and classify.

In [ ]:
daily = daily_performance(out)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily.index, daily["pi"], ".", ms=3, alpha=0.5, label="daily PI")
ax.plot(daily.index, daily["pi"].rolling(30, min_periods=10).median(), lw=2, label="30-day median")
ax.axhline(1.0, c="gray", ls="--")
ax.set(ylim=(0, 1.4), ylabel="Performance index", title=f"{site.name}: daily performance index")
ax.legend();

In [ ]:
# Worst days: candidates for the first labelled events
daily.sort_values("pi").head(15)

## Notes / findings

*Write down what you find here. These become the README write-up and the talking points for the demo.*

-